# HEMM Dataset FINAL — Exploratory Data Analysis (EDA)
## NALCO Internship Project
### Dataset: HEMM_Dataset_FINAL.csv | 93 Columns | 2000 Rows
---
**12 EDA Graphs covering:**
- Equipment failure analysis
- All 10 sensor readings vs failure
- Upper/Lower limits visualization
- Failure types and components
- Maintenance priority analysis
- HEMM parts condition
- Part life remaining %
- Correlation heatmap
- Health score analysis
- Maintenance cost and downtime
- Monthly failure trends
- Replacement needed summary

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# Create folders
os.makedirs('../graphs', exist_ok=True)

# Load dataset
df = pd.read_csv('../data/HEMM_Dataset_FINAL.csv')
df['Date'] = pd.to_datetime(df['Date'])
df['Failure_Type']      = df['Failure_Type'].fillna('None')
df['Failure_Component'] = df['Failure_Component'].fillna('None')

# Color palette
C = {
    'main'  : '#2563EB',
    'fail'  : '#DC2626',
    'ok'    : '#16A34A',
    'warn'  : '#D97706',
    'purple': '#7C3AED',
    'teal'  : '#0D9488',
    'pink'  : '#EC4899',
    'orange': '#F97316',
}

print("Dataset loaded!")
print(f"Rows         : {df.shape[0]}")
print(f"Columns      : {df.shape[1]}")
print(f"Failure Rate : {df['Failure'].mean()*100:.1f}%")
print(f"Equipment Types: {df['Equipment_Type'].unique().tolist()}")
df.head(3)


## Graph 1 — Failure Count & Rate by Equipment Type
**Which HEMM machine fails the most?**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Graph 1 — Failure Count & Rate by Equipment Type', fontsize=14, fontweight='bold')

fc = df.groupby('Equipment_Type')['Failure'].sum().sort_values(ascending=False)
axes[0].bar(fc.index, fc.values, color=C['fail'], edgecolor='white')
axes[0].set_title('Total Failures by Equipment Type', fontweight='bold')
axes[0].set_xlabel('Equipment Type'); axes[0].set_ylabel('Failures')
axes[0].tick_params(axis='x', rotation=30)
for i,v in enumerate(fc.values):
    axes[0].text(i, v+0.5, str(v), ha='center', fontsize=9, fontweight='bold')

fr = df.groupby('Equipment_Type')['Failure'].mean().mul(100).sort_values(ascending=False)
axes[1].bar(fr.index, fr.values, color=C['warn'], edgecolor='white')
axes[1].axhline(df['Failure'].mean()*100, color='red', linestyle='--',
                label=f'Avg: {df["Failure"].mean()*100:.1f}%')
axes[1].set_title('Failure Rate % by Equipment Type', fontweight='bold')
axes[1].set_xlabel('Equipment Type'); axes[1].set_ylabel('Failure Rate (%)')
axes[1].tick_params(axis='x', rotation=30); axes[1].legend()
for i,v in enumerate(fr.values):
    axes[1].text(i, v+0.2, f'{v:.1f}%', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('../graphs/HEMM_G1_failure_by_equipment_type.png', dpi=130, bbox_inches='tight')
plt.show()
print(f"Most failing equipment : {fc.index[0]} ({fc.values[0]} failures)")
print(f"Highest failure rate   : {fr.index[0]} ({fr.values[0]:.1f}%)")


## Graph 2 — All 10 Sensor Readings: Normal vs Failure
**Which sensor shows the biggest difference between normal and failure?**

In [ ]:
sensors = ['Engine_Temp_C','Oil_Pressure_bar','Vibration_mms','Fuel_Consumption_Lhr',
           'Tyre_Pressure_PSI','Coolant_Level','Battery_Voltage_V',
           'Hydraulic_Pressure_bar','Exhaust_Temp_C','RPM']
labels  = ['Engine Temp (C)','Oil Pressure (bar)','Vibration (mm/s)','Fuel (L/hr)',
           'Tyre PSI','Coolant Level','Battery (V)','Hydraulic (bar)','Exhaust Temp (C)','RPM']

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle('Graph 2 — All 10 Sensors: Normal vs Failure', fontsize=14, fontweight='bold')

for ax, col, lbl in zip(axes.flatten(), sensors, labels):
    bp = ax.boxplot([df[df['Failure']==0][col], df[df['Failure']==1][col]],
                   labels=['Normal','Failure'], patch_artist=True)
    bp['boxes'][0].set_facecolor(C['ok']);  bp['boxes'][0].set_alpha(0.7)
    bp['boxes'][1].set_facecolor(C['fail']); bp['boxes'][1].set_alpha(0.7)
    ax.set_title(lbl, fontsize=9, fontweight='bold')
    n = df[df['Failure']==0][col].mean()
    f = df[df['Failure']==1][col].mean()
    ax.set_xlabel(f'Normal:{n:.1f}  Fail:{f:.1f}', fontsize=8)

plt.tight_layout()
plt.savefig('../graphs/HEMM_G2_all_sensors_vs_failure.png', dpi=130, bbox_inches='tight')
plt.show()


## Graph 3 — Sensor Readings vs Upper/Lower Limits
**How many readings are outside the safe operating range?**

In [ ]:
limit_pairs = [
    ('Engine_Temp_C','Engine_Temp_C_Lower_Limit','Engine_Temp_C_Upper_Limit','Engine Temp (C)'),
    ('Oil_Pressure_bar','Oil_Pressure_bar_Lower_Limit','Oil_Pressure_bar_Upper_Limit','Oil Pressure (bar)'),
    ('Vibration_mms','Vibration_mms_Lower_Limit','Vibration_mms_Upper_Limit','Vibration (mm/s)'),
    ('Fuel_Consumption_Lhr','Fuel_Consumption_Lhr_Lower_Limit','Fuel_Consumption_Lhr_Upper_Limit','Fuel (L/hr)'),
    ('Tyre_Pressure_PSI','Tyre_Pressure_PSI_Lower_Limit','Tyre_Pressure_PSI_Upper_Limit','Tyre PSI'),
    ('Coolant_Level','Coolant_Level_Lower_Limit','Coolant_Level_Upper_Limit','Coolant Level'),
    ('Battery_Voltage_V','Battery_Voltage_V_Lower_Limit','Battery_Voltage_V_Upper_Limit','Battery (V)'),
    ('Hydraulic_Pressure_bar','Hydraulic_Pressure_bar_Lower_Limit','Hydraulic_Pressure_bar_Upper_Limit','Hydraulic (bar)'),
    ('Exhaust_Temp_C','Exhaust_Temp_C_Lower_Limit','Exhaust_Temp_C_Upper_Limit','Exhaust Temp (C)'),
    ('RPM','RPM_Lower_Limit','RPM_Upper_Limit','RPM'),
]

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle('Graph 3 — Sensor Readings vs Upper/Lower Limits (Green=Lower Limit, Red=Upper Limit)',
             fontsize=13, fontweight='bold')

for ax,(col,lo_col,hi_col,lbl) in zip(axes.flatten(), limit_pairs):
    lo = df[lo_col].iloc[0]; hi = df[hi_col].iloc[0]
    ax.hist(df[col], bins=30, color=C['main'], alpha=0.6, edgecolor='white')
    ax.axvline(lo, color='green', linestyle='--', linewidth=2, label=f'LL:{lo}')
    ax.axvline(hi, color='red',   linestyle='--', linewidth=2, label=f'UL:{hi}')
    ax.set_title(lbl, fontsize=9, fontweight='bold')
    ax.legend(fontsize=7)
    below = (df[col] < lo).sum()
    above = (df[col] > hi).sum()
    ax.set_xlabel(f'Below LL:{below}  Above UL:{above}', fontsize=8)

plt.tight_layout()
plt.savefig('../graphs/HEMM_G3_sensor_limits.png', dpi=130, bbox_inches='tight')
plt.show()


## Graph 4 — Failure Type & Component
**Which failure type and which HEMM part fails most?**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Graph 4 — Failure Type & Component Distribution', fontsize=14, fontweight='bold')

ft = df[df['Failure']==1]['Failure_Type'].value_counts()
axes[0].barh(ft.index, ft.values, color=C['purple'], edgecolor='white')
axes[0].set_title('Count of Each Failure Type', fontweight='bold')
axes[0].set_xlabel('Number of Failures')
for i,v in enumerate(ft.values):
    axes[0].text(v+0.3, i, str(v), va='center', fontsize=9, fontweight='bold')

fc2 = df[df['Failure']==1]['Failure_Component'].value_counts()
pie_c = [C['fail'],C['warn'],C['purple'],C['teal'],C['main'],C['pink'],C['orange']]
axes[1].pie(fc2.values, labels=fc2.index, autopct='%1.1f%%',
            colors=pie_c[:len(fc2)], startangle=90,
            wedgeprops={'edgecolor':'white','linewidth':1.5})
axes[1].set_title('Which HEMM Part Fails Most (%)', fontweight='bold')

plt.tight_layout()
plt.savefig('../graphs/HEMM_G4_failure_types.png', dpi=130, bbox_inches='tight')
plt.show()
print(f"Most common failure type      : {ft.index[0]} ({ft.values[0]} times)")
print(f"Most failing HEMM component   : {fc2.index[0]} ({fc2.values[0]/fc2.sum()*100:.1f}%)")


## Graph 5 — Maintenance Priority Analysis
**How urgent is maintenance for each equipment?**

In [ ]:
mp_order    = ['Low','Medium','High','Critical']
bar_colors  = [C['ok'], C['warn'], C['fail'], '#7F1D1D']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Graph 5 — Maintenance Priority Analysis', fontsize=14, fontweight='bold')

mp_counts = df['Maintenance_Priority'].value_counts().reindex(mp_order, fill_value=0)
axes[0].bar(mp_counts.index, mp_counts.values, color=bar_colors, edgecolor='white')
axes[0].set_title('Equipment Count by Maintenance Priority', fontweight='bold')
axes[0].set_ylabel('Count')
for i,v in enumerate(mp_counts.values):
    axes[0].text(i, v+5, str(v), ha='center', fontsize=11, fontweight='bold')

mp_fail = df.groupby('Maintenance_Priority')['Failure'].mean().mul(100).reindex(mp_order)
axes[1].bar(mp_fail.index, mp_fail.values, color=bar_colors, edgecolor='white')
axes[1].set_title('Failure Rate % by Maintenance Priority', fontweight='bold')
axes[1].set_ylabel('Failure Rate (%)')
for i,v in enumerate(mp_fail.values):
    axes[1].text(i, v+0.5, f'{v:.1f}%', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('../graphs/HEMM_G5_maintenance_priority.png', dpi=130, bbox_inches='tight')
plt.show()
print(f"Critical machines: {mp_counts.get('Critical',0)}")
print(f"High priority    : {mp_counts.get('High',0)}")


## Graph 6 — HEMM Parts Condition
**What is the current condition of each part? (0=Good, 1=Warning, 2=Critical)**

In [ ]:
part_cols   = ['Engine_Condition_enc','Tyre_Condition_enc','Hydraulic_Condition_enc',
               'Brake_Condition_enc','Electrical_Condition_enc','Fuel_System_Condition_enc',
               'Transmission_Condition_enc','Cooling_System_Condition_enc']
part_labels = ['Engine','Tyre','Hydraulic','Brake','Electrical','Fuel System','Transmission','Cooling']
cond_colors = [C['ok'], C['warn'], C['fail']]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle('Graph 6 — HEMM Parts Condition (0=Good  1=Warning  2=Critical)', fontsize=14, fontweight='bold')

for ax, col, lbl in zip(axes.flatten(), part_cols, part_labels):
    vc = df[col].value_counts().sort_index()
    ax.bar([str(int(x)) for x in vc.index], vc.values,
           color=[cond_colors[int(i)] for i in vc.index], edgecolor='white')
    ax.set_title(lbl, fontweight='bold', fontsize=10)
    ax.set_xlabel('0=Good  1=Warning  2=Critical', fontsize=8)
    ax.set_ylabel('Count')
    for i,v in enumerate(vc.values):
        ax.text(i, v+2, str(v), ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('../graphs/HEMM_G6_parts_condition.png', dpi=130, bbox_inches='tight')
plt.show()


## Graph 7 — Part Life Remaining %
**How much life is left in each HEMM component?**

In [ ]:
life_cols   = ['Engine_Life_Remaining_pct','Tyre_Life_Remaining_pct',
               'Hydraulic_Life_Remaining_pct','Battery_Life_Remaining_pct','Brake_Life_Remaining_pct']
life_labels = ['Engine','Tyre','Hydraulic','Battery','Brake']
life_colors = [C['main'],C['orange'],C['teal'],C['purple'],C['warn']]

fig, axes = plt.subplots(1, 5, figsize=(20, 5))
fig.suptitle('Graph 7 — Part Life Remaining % (Red line = Average)', fontsize=14, fontweight='bold')

for ax, col, lbl, clr in zip(axes, life_cols, life_labels, life_colors):
    ax.hist(df[col], bins=25, color=clr, alpha=0.8, edgecolor='white')
    avg = df[col].mean()
    ax.axvline(avg, color='red', linestyle='--', linewidth=2, label=f'Avg:{avg:.1f}%')
    ax.set_title(f'{lbl} Life %', fontweight='bold', fontsize=10)
    ax.set_xlabel('Life Remaining (%)'); ax.legend(fontsize=8)
    low_count = (df[col] < 30).sum()
    ax.set_ylabel(f'Count (Below 30%: {low_count})')

plt.tight_layout()
plt.savefig('../graphs/HEMM_G7_part_life_remaining.png', dpi=130, bbox_inches='tight')
plt.show()


## Graph 8 — Correlation Heatmap
**Which sensor is most strongly linked to failure?**

In [ ]:
fig, ax = plt.subplots(figsize=(13, 10))
fig.suptitle('Graph 8 — Correlation Heatmap', fontsize=14, fontweight='bold')

corr_cols = ['Engine_Temp_C','Oil_Pressure_bar','Vibration_mms','Fuel_Consumption_Lhr',
             'Tyre_Pressure_PSI','Coolant_Level','Battery_Voltage_V',
             'Hydraulic_Pressure_bar','Exhaust_Temp_C','RPM',
             'Operating_Hours','Health_Score','Days_to_Next_Failure','Failure']

mask = np.triu(np.ones(len(corr_cols), dtype=bool))
sns.heatmap(df[corr_cols].corr(), annot=True, fmt='.2f', cmap='RdYlGn',
            mask=mask, ax=ax, linewidths=0.5, annot_kws={'size':9}, vmin=-1, vmax=1)
ax.set_title('Red=Negative | Green=Positive | Closer to 1 = Stronger link to Failure', fontsize=10)

plt.tight_layout()
plt.savefig('../graphs/HEMM_G8_correlation_heatmap.png', dpi=130, bbox_inches='tight')
plt.show()

print("Top 5 features correlated with Failure:")
corr_with_failure = df[corr_cols].corr()['Failure'].drop('Failure').abs().sort_values(ascending=False)
for feat, val in corr_with_failure.head(5).items():
    print(f"  {feat:35s}: {val:.3f}")


## Graph 9 — Health Score & Operating Hours
**Do machines with more hours and lower health score fail more?**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Graph 9 — Health Score & Operating Hours vs Failure', fontsize=14, fontweight='bold')

for val,lbl,clr in [(0,'Normal',C['ok']),(1,'Failure',C['fail'])]:
    axes[0].hist(df[df['Failure']==val]['Health_Score'], bins=25,
                 alpha=0.6, label=lbl, color=clr, edgecolor='white')
axes[0].set_title('Health Score: Normal vs Failure', fontweight='bold')
axes[0].set_xlabel('Health Score (0=Very Bad, 100=Perfect)'); axes[0].legend()

axes[1].scatter(df[df['Failure']==0]['Operating_Hours'],
                df[df['Failure']==0]['Health_Score'],
                alpha=0.2, color=C['ok'], s=12, label='Normal')
axes[1].scatter(df[df['Failure']==1]['Operating_Hours'],
                df[df['Failure']==1]['Health_Score'],
                alpha=0.5, color=C['fail'], s=18, label='Failure')
axes[1].set_title('Operating Hours vs Health Score', fontweight='bold')
axes[1].set_xlabel('Total Operating Hours'); axes[1].set_ylabel('Health Score'); axes[1].legend()

plt.tight_layout()
plt.savefig('../graphs/HEMM_G9_health_operating_hours.png', dpi=130, bbox_inches='tight')
plt.show()
print(f"Avg health score — Normal : {df[df['Failure']==0]['Health_Score'].mean():.1f}")
print(f"Avg health score — Failure: {df[df['Failure']==1]['Health_Score'].mean():.1f}")


## Graph 10 — Maintenance Cost & Downtime
**How much does each maintenance level cost and how long does it take?**

In [ ]:
mp_order   = ['Low','Medium','High','Critical']
bar_colors = [C['ok'], C['warn'], C['fail'], '#7F1D1D']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Graph 10 — Maintenance Cost & Downtime by Priority', fontsize=14, fontweight='bold')

mp_cost = df.groupby('Maintenance_Priority')['Maintenance_Cost_INR'].mean().reindex(mp_order)
axes[0].bar(mp_cost.index, mp_cost.values, color=bar_colors, edgecolor='white')
axes[0].set_title('Average Maintenance Cost (INR)', fontweight='bold')
axes[0].set_ylabel('Cost (INR)')
for i,v in enumerate(mp_cost.values):
    axes[0].text(i, v+200, f'Rs.{v:,.0f}', ha='center', fontsize=9, fontweight='bold')

mp_down = df.groupby('Maintenance_Priority')['Estimated_Downtime_Hrs'].mean().reindex(mp_order)
axes[1].bar(mp_down.index, mp_down.values, color=bar_colors, edgecolor='white')
axes[1].set_title('Average Estimated Downtime (Hours)', fontweight='bold')
axes[1].set_ylabel('Downtime (Hours)')
for i,v in enumerate(mp_down.values):
    axes[1].text(i, v+0.2, f'{v:.1f} hrs', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('../graphs/HEMM_G10_cost_downtime.png', dpi=130, bbox_inches='tight')
plt.show()
print(f"Critical maintenance avg cost    : Rs.{mp_cost.get('Critical',0):,.0f}")
print(f"Critical maintenance avg downtime: {mp_down.get('Critical',0):.1f} hours")


## Graph 11 — Monthly Trend & Maintenance History
**Is failure increasing over time? Do machines fail more when maintenance is overdue?**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Graph 11 — Monthly Failure Trend & Maintenance History', fontsize=14, fontweight='bold')

monthly = df.groupby(['Year','Month'])['Failure'].sum().reset_index()
monthly['Period'] = monthly['Year'].astype(str)+'-'+monthly['Month'].astype(str).str.zfill(2)
axes[0].plot(range(len(monthly)), monthly['Failure'],
             color=C['main'], marker='o', markersize=4, linewidth=2)
step = max(1, len(monthly)//8)
axes[0].set_xticks(range(0, len(monthly), step))
axes[0].set_xticklabels(monthly['Period'].iloc[::step], rotation=45, fontsize=8)
axes[0].set_title('Monthly Failure Count (2021–2024)', fontweight='bold')
axes[0].set_ylabel('Number of Failures'); axes[0].grid(True, alpha=0.3)

for val,lbl,clr in [(0,'Normal',C['ok']),(1,'Failure',C['fail'])]:
    axes[1].hist(df[df['Failure']==val]['Days_Since_Last_Maintenance'],
                 bins=30, alpha=0.6, label=lbl, color=clr, edgecolor='white')
axes[1].set_title('Days Since Last Maintenance: Normal vs Failure', fontweight='bold')
axes[1].set_xlabel('Days Since Last Maintenance'); axes[1].legend()

plt.tight_layout()
plt.savefig('../graphs/HEMM_G11_monthly_trend.png', dpi=130, bbox_inches='tight')
plt.show()


## Graph 12 — Replacement Needed & Days to Next Failure
**Which parts need replacement and how soon will failure happen?**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Graph 12 — Replacement Needed & Days to Next Failure', fontsize=14, fontweight='bold')

repl_cols   = ['Engine_Replacement_Needed_enc','Tyre_Replacement_Needed_enc',
               'Hydraulic_Replacement_Needed_enc','Brake_Replacement_Needed_enc',
               'Electrical_Replacement_Needed_enc']
repl_labels = ['Engine','Tyre','Hydraulic','Brake','Electrical']
yes_counts  = [(df[col]==2).sum() for col in repl_cols]
mon_counts  = [(df[col]==1).sum() for col in repl_cols]
x = range(len(repl_labels))
axes[0].bar(x, yes_counts, color=C['fail'], label='Replace Now', edgecolor='white')
axes[0].bar(x, mon_counts, bottom=yes_counts, color=C['warn'], label='Monitor', edgecolor='white')
axes[0].set_xticks(x); axes[0].set_xticklabels(repl_labels)
axes[0].set_title('Parts Needing Replacement or Monitoring', fontweight='bold')
axes[0].set_ylabel('Count'); axes[0].legend()
for i,(y,m) in enumerate(zip(yes_counts,mon_counts)):
    if y>0: axes[0].text(i, y/2, str(y), ha='center', color='white', fontsize=9, fontweight='bold')

for val,lbl,clr in [(0,'Normal',C['ok']),(1,'Failure',C['fail'])]:
    axes[1].hist(df[df['Failure']==val]['Days_to_Next_Failure'],
                 bins=25, alpha=0.6, label=lbl, color=clr, edgecolor='white')
axes[1].set_title('Days to Next Failure Distribution', fontweight='bold')
axes[1].set_xlabel('Days'); axes[1].set_ylabel('Count'); axes[1].legend()

plt.tight_layout()
plt.savefig('../graphs/HEMM_G12_replacement_days_to_failure.png', dpi=130, bbox_inches='tight')
plt.show()
print("Parts needing immediate replacement:")
for lbl, yes in zip(repl_labels, yes_counts):
    print(f"  {lbl:15s}: {yes} machines")


## EDA Summary

In [ ]:
print("=" * 60)
print("HEMM FINAL DATASET — EDA SUMMARY")
print("=" * 60)
print(f"Total Records         : {len(df)}")
print(f"Total Failures        : {df['Failure'].sum()}")
print(f"Overall Failure Rate  : {df['Failure'].mean()*100:.1f}%")
print()
print(f"Most failing equipment : {df.groupby('Equipment_Type')['Failure'].sum().idxmax()}")
print(f"Most common failure    : {df[df['Failure']==1]['Failure_Type'].value_counts().index[0]}")
print(f"Most failing component : {df[df['Failure']==1]['Failure_Component'].value_counts().index[0]}")
print()
print("Sensor avg — Normal vs Failure:")
for col in ['Engine_Temp_C','Vibration_mms','Oil_Pressure_bar']:
    n = df[df['Failure']==0][col].mean()
    f = df[df['Failure']==1][col].mean()
    print(f"  {col:25s}: Normal={n:.1f}  Failure={f:.1f}")
print()
print("Maintenance urgency:")
mp = df['Maintenance_Priority'].value_counts()
for p in ['Critical','High','Medium','Low']:
    print(f"  {p:10s}: {mp.get(p,0)} machines")
print()
print("EDA Complete! Ready for ML Model.")
